In [2]:
import numpy as np
import matplotlib.pyplot as plt
import fuzzylite as fl
from fuzzylite import settings

# Register custom hedges to match lecture slide definitions
settings.factory_manager.hedge.constructors["slightly"] = lambda: fl.HedgeLambda("slightly", lambda x: x ** 1.7)
settings.factory_manager.hedge.constructors["more_or_less"] = lambda: fl.HedgeLambda("more_or_less", lambda x: x ** 0.5)
settings.factory_manager.hedge.constructors["a_little"] = lambda: fl.HedgeLambda("a_little", lambda x: x ** 1.3)
settings.factory_manager.hedge.constructors["extremely"] = lambda: fl.HedgeLambda("extremely", lambda x: x ** 3)
settings.factory_manager.hedge.constructors["very_very"] = lambda: fl.HedgeLambda("very_very", lambda x: x ** 4)

print(f"pyfuzzylite version: {fl.__version__}")
print(f"Available hedges: {list(settings.factory_manager.hedge.constructors.keys())}")

pyfuzzylite version: 8.0.6
Available hedges: ['any', 'extremely', 'not', 'seldom', 'somewhat', 'very', 'slightly', 'more_or_less', 'a_little', 'very_very']


In [3]:
def get_output(engine, name):
    """
    Extract a scalar float from a fuzzy engine output variable.
    pyfuzzylite may return a numpy array instead of a plain float.
    This helper handles both cases safely.
    """
    val = engine.output_variable(name).value
    return float(val.item()) if hasattr(val, 'item') else float(val)

def plot_membership_functions(variable, title=None, ax=None):
    """Plot all membership functions for a pyfuzzylite variable."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 3.5))
    x = np.linspace(variable.minimum, variable.maximum, 500)
    for term in variable.terms:
        y = np.array([float(term.membership(xi)) for xi in x])
        ax.plot(x, y, linewidth=2, label=term.name)
    ax.set_xlabel(variable.name)
    ax.set_ylabel('Membership Degree')
    ax.set_title(title or f'Membership Functions: {variable.name}')
    ax.legend(loc='best', fontsize=9)
    ax.set_ylim(-0.05, 1.1)
    ax.grid(True, alpha=0.3)
    return ax

def plot_simulation(times, speeds, throttles, errors, target, title="Cruise Control Response"):
    """Plot a 3-panel time-domain response for a control simulation."""
    fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=True)
    axes[0].plot(times, speeds, 'b-', linewidth=1.5, label='Actual Speed')
    axes[0].axhline(y=target, color='r', linestyle='--', linewidth=1, label=f'Target ({target} km/h)')
    axes[0].set_ylabel('Speed (km/h)')
    axes[0].legend(loc='best')
    axes[0].set_title(title)
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(times, throttles, 'g-', linewidth=1.5)
    axes[1].axhline(y=0, color='gray', linestyle=':', linewidth=0.5)
    axes[1].set_ylabel('Throttle')
    axes[1].grid(True, alpha=0.3)

    axes[2].plot(times, errors, 'r-', linewidth=1.5)
    axes[2].axhline(y=0, color='gray', linestyle=':', linewidth=0.5)
    axes[2].set_ylabel('Speed Error (km/h)')
    axes[2].set_xlabel('Time (seconds)')
    axes[2].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

def plot_comparison(times, speed_dict, target, title="Controller Comparison"):
    """Overlay multiple speed traces for comparison."""
    plt.figure(figsize=(10, 5))
    for label, speeds in speed_dict.items():
        plt.plot(times, speeds, linewidth=1.5, label=label)
    plt.axhline(y=target, color='r', linestyle='--', linewidth=1, label=f'Target ({target} km/h)')
    plt.xlabel('Time (seconds)')
    plt.ylabel('Speed (km/h)')
    plt.title(title)
    plt.legend(loc='best')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

def plot_control_surface(engine, input1_name, input2_name, output_name, resolution=50):
    """Generate and plot the 3D control surface for a 2-input, 1-output fuzzy system."""
    iv1 = engine.input_variable(input1_name)
    iv2 = engine.input_variable(input2_name)

    x1 = np.linspace(iv1.minimum, iv1.maximum, resolution)
    x2 = np.linspace(iv2.minimum, iv2.maximum, resolution)
    X1, X2 = np.meshgrid(x1, x2)
    Z = np.zeros_like(X1)

    for i in range(len(x2)):
        for j in range(len(x1)):
            iv1.value = float(x1[j])
            iv2.value = float(x2[i])
            engine.process()
            Z[i, j] = get_output(engine, output_name)

    fig = plt.figure(figsize=(10, 7))
    ax = fig.add_subplot(111, projection='3d')
    surf = ax.plot_surface(X1, X2, Z, cmap='RdYlGn', edgecolor='none', alpha=0.85)
    ax.set_xlabel(input1_name)
    ax.set_ylabel(input2_name)
    ax.set_zlabel(output_name)
    ax.set_title(f'Control Surface: {engine.name}')
    fig.colorbar(surf, shrink=0.5, aspect=10, label=output_name)
    plt.tight_layout()
    plt.show()
    
    return X1, X2, Z

print("Plotting helpers loaded.")


Plotting helpers loaded.
